In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Updated Label Mapping
label_map = {
    'benign': 0, 'adware': 1, 'backdoor': 2, 'banker': 3, 'dropper': 4,
    'fileinjector': 5, 'NoCategory': 6, 'PUA': 7, 'Ransomware': 8,
    'Riskware': 9, 'SMS': 10, 'Scareware': 11, 'Spy': 12, 'Trojan': 13, 'Zeroday': 14
}

# Load datasets (Replace 'malware_multiclass.csv' with your actual multiclass filename)
benign_df = pd.read_csv("processedData/benign.csv")
mal_df = pd.read_csv("processedData/malware_multiclass.csv") 

# Reset columns to ensure consistency before concatenation
benign_df.columns = range(benign_df.shape[1])
mal_df.columns = range(mal_df.shape[1])

# Concatenate datasets
df = pd.concat([benign_df, mal_df], axis=0, ignore_index=True)

# Ensure feature 9504 contains the integer values 0-14 according to label_map
print("Value counts for multiclass labels:")
print(df[9504].value_counts())

In [ ]:
df = df.fillna(0)


X = df.drop(9504, axis=1).select_dtypes(include=[np.number])  # also drop hash column (1)
y = df[9504]

# Compute correlation of each feature with label
correlations = X.corrwith(y)

# Sort by absolute correlation
corr_sorted = correlations.abs().sort_values(ascending=False)

print(corr_sorted.head(20))

In [ ]:
"""
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X, y)

mi_scores = pd.Series(mi, index=X.columns)
top_features = mi_scores.sort_values(ascending=False).head(50)
"""

In [ ]:
TOP_N = 100

top_features = corr_sorted.head(TOP_N).index.tolist()

print("Selected feature indices:", top_features)

In [ ]:


#top_features = mi_scores.sort_values(ascending=False).head(20).index
corr_matrix = df[top_features + [9504]].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm")
plt.title("Top features correlations with label")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_reduced = X[top_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

#model - RandomForestClassifier with hyperparameters tuned for multiclass classification
model = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
target_names = [label_map[i] for i in sorted(label_map.keys())]

print("Multiclass Accuracy:", model.score(X_test, y_test))
print("\nClassification Report:")
inv_label_map = {v: k for k, v in label_map.items()}
target_names = [inv_label_map[i] for i in sorted(inv_label_map.keys())]
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
#cross validation

from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_reduced, y, cv=5)
print(scores.mean())

In [ ]:
# Detailed classification report with target names

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
import shap
from sklearn.ensemble import RandomForestClassifier

rf_shap = RandomForestClassifier(
    n_estimators=50, 
    max_depth=10, # Limited depth for faster SHAP calculation
    random_state=42, 
    n_jobs=-1
)
rf_shap.fit(X_train, y_train)

# 2. Initialize the Explainer
# TreeExplainer is optimized specifically for Random Forests
explainer = shap.TreeExplainer(rf_shap)

# 3. Generate SHAP values for a sample
# We use a sample because calculating SHAP for the whole test set is slow
X_sample = X_test.sample(500)
shap_explanations = explainer(X_sample)
shap_values = shap_explanations.values

In [ ]:
target_class_index = 0
print(f"Generating Detailed Beeswarm for Class {target_class_index}...")
shap.plots.beeswarm(shap_explanations[:, :, target_class_index])

In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_sample, plot_type="bar")

In [ ]:
inv_label_map = {v: k for k, v in label_map.items()}



# 2. Select the app and the class you want to explain
i = 0  # Index of the Android app in X_sample


for idx in range(len(inv_label_map)): 
 target_class_index = idx # Example: Explain why it might be 'adware'

# 3. Prepare data for the plot
# Using .astype(float) prevents the UFuncTypeError by ensuring numbers aren't objects
 features_row = X_sample.iloc[i, :]
 feature_names = X_sample.columns.tolist()
 features_display = [str(val) for val in features_row.values]
 class_name = inv_label_map[target_class_index]

# 4. Generate the Force Plot
 print(f"Generating explanation for Class: {class_name} (Index {target_class_index})")

 shap.force_plot(
    float(explainer.expected_value[target_class_index]), 
    shap_values[i, :, target_class_index], 
    features_display,
    feature_names=feature_names,
    out_names=class_name, # This labels the final value as the specific malware type
    matplotlib=True,
    show=False
 )

# 5. Clean up the title and layout
 plt.title(f"SHAP Explanation for {class_name.upper()}", fontsize=10, pad=18)
 plt.tight_layout()
 plt.show()


In [ ]:
sample = X_test.sample(n=100, random_state=42).reset_index(drop=True)
pred = model.predict(sample)

# 1. Setup the explainer and explanations
explainer = shap.TreeExplainer(model)
# Note: For some RF versions, you might need to use shap_values = explainer.shap_values(sample)
# but the standard explainer call is usually preferred:
shap_explanations = explainer(sample, check_additivity=False)

# Inverse label map to show names instead of numbers
inv_label_map = {v: k for k, v in label_map.items()}

# 2. Iterate through the predictions
for i, app_pred_idx in enumerate(pred):
    # Get the human-readable name of the predicted class
    predicted_class_name = inv_label_map[app_pred_idx]
    
    # DYNAMIC INDEXING: 
    # Get the SHAP values for the specific class that was PREDICTED
    # shap_explanations.values shape is (samples, features, classes)
    current_shap_values = shap_explanations.values[i, :, app_pred_idx]
    feature_names = sample.columns
    
    # Sort features by their impact (top 3)
    top_indices = np.argsort(np.abs(current_shap_values))[-3:][::-1]
    
    reasons = []
    for idx in top_indices:
        val = current_shap_values[idx]
        feature_name = feature_names[idx]
        
        # If SHAP is positive, it increased the probability of this class
        # If SHAP is negative, it decreased the probability
        if val > 0:
            direction = f"increased evidence for {predicted_class_name}"
        else:
            direction = f"decreased evidence for {predicted_class_name}"
            
        reasons.append(f"{feature_name} ({direction} by {abs(val):.4f})")
        
    print(f"App {i}: Predicted as [{predicted_class_name.upper()}]")
    print(f"   Top Reasons: {', '.join(reasons)}\n")

In [ ]:
from xgboost import XGBClassifier

modelx = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method="hist",
    n_jobs=-1
)

modelx.fit(X_train, y_train)

# Evaluation
y_pred = modelx.predict(X_test)
target_names = [label_map[i] for i in sorted(label_map.keys())]

print("Multiclass Accuracy:", modelx.score(X_test, y_test))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))